In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import scipy.stats

pandas.set_option("future.no_silent_downcasting", True)

# Exploration de données : dataset [AMES](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

Dans ces travaux pratiques, nous allons nous intéresser à la compréhension d'un des datasets qui nous servira de terrain de jeu à certains points de la formation.

Ce dataset contient tous types de variables, a des valeurs manquantes, un nombre important de features et celles-ci ne sont pas forcément adaptées aux méthodes qu'on essaye de leur appliquer : il modélise bien un dataset que vous rencontrerez au jour le jour en faisant de la data science.

Aujourd'hui, nous allons suivre ces étapes :

1. obtenir les données
2. nettoyer les données
3. explorer les données


## Définition du problème

La problématique d'intérêt dans cette analyse est la prédiction de prix de maisons en Iowa, aux États-Unis.

Pour que le problème soit bien posé, il faut établir dès le début de l'analyse quelles sont les sorties attendues et les mesures d'évaluation de ces sorties.

- *Définissez la sortie du modèle.*

- *Proposez une ou plusieurs métriques pour évaluer la qualité de ce que le modèle prédira.*

- *Quel est le problème qui se pose si on ne définit pas ces deux éléments en amont ?*

### Solution

- la sortie du modèle est le prix de la maison
- différence entre la prédiction et la réalité, potentiellement au carré
- on introduit un biais statistique : on pourrait choisir les métriques a posteri qui maximisent nos résultats

## Récupération des données

Une fois le repository git des données cloné, les données se trouvent dans le répertoire `dataset-ames/`. Celui-ci contient en particulier un fichier `train.csv` que vous pouvez utiliser comme ensemble d'entraînement et de validation et un fichier `test.csv` qui simule une utilisation en production : on ne dispose pas des targets.

Conseils :

- [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) vous sera utile. La colonne d'index des données est nommée `Id` dans le CSV, vous pouvez donc passer l'argument `index_col="Id"` à la méthode en plus du chemin du CSV.

- N'hésitez pas à consulter [la documentation des `DataFrame`s pandas](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html).

Exercice :

- *Affichez le nom des colonnes des dataframes ainsi que la dimension des données (nombre de lignes, nombre de colonnes)*.

- *Une description des variables du csv sont décrites dans le fichier data_description.txt. Affichez-le dans une cellule (ou téléchargez-le)*.

In [ ]:
from google.colab import files

!git clone https://github.com/nzmonzmp/dataset-ames.git
!ls -l dataset-ames/
files.download("dataset-ames/data_description.txt")

In [ ]:
# train_df = pandas.???
# test_df = pandas.???

### Solution

In [ ]:
train_df = pandas.read_csv("dataset-ames/train.csv", index_col=["Id"])
test_df = pandas.read_csv("dataset-ames/test.csv", index_col="Id")

print(f"Colonnes : {', '.join(train_df.columns)}")
print(f"Forme de la DataFrame de train : {train_df.shape}")
print(f"Forme de la DataFrame de test : {test_df.shape}")

In [ ]:
train_df

## Extraction des variables

*Créez les variables suivantes :*

- *`train_X` qui contient toutes les colonnes de `train_df` sauf `SalePrice`. Faites une sélection ou utilisez directement [`DataFrame.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html?highlight=drop#pandas.DataFrame.drop)*
- *`test_X` qui contient toutes les colonnes de `test_df`*
- *`train_y` qui contient `SalePrice` de `train_df`*

*Créez aussi la variable `all_X` en utilisant [`pandas.concat`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html?highlight=concat#pandas.concat), concaténation de `train_X` et `test_X` qui nous permettra de tout prétraiter de manière homogène.*

*__Attention, utiliser `test_X` (et donc `all_X`) peut induire un biais statistique, il faut utiliser seulement `train_X` quand on utilise les données pour dériver une valeur !__*

In [ ]:
# train_X = ???
# train_y = ???
# test_X = ???
# all_X = ???

### Solution

In [ ]:
train_X = train_df.drop(columns="SalePrice")
train_y = train_df["SalePrice"]
test_X = test_df
all_X = pandas.concat([train_X, test_X])

print("Forme des données d'entrainement :", train_X.shape, train_y.shape)
print("Forme des données de test :", test_X.shape)
print("Concaténation : ", all_X.shape)

In [ ]:
train_df[["SalePrice", "MSSubClass"]]

## Nettoyage des données

Cette étape, jointe avec l'exploration, passe par (au moins) 4 étapes :

- gérer les valeurs manquantes
- préprocessing (texte, image, …)
- standardisation
- transformation

Ici, nous n'aurons pas besoin de préprocessing spécifique, car les données sont principalement catégorielles ou continues, sans données textuelles ou d'images.

### Valeurs manquantes

Les différences de format, les conditions de récolte des données ainsi que beaucoup d'autres facteurs entraînent de nombreuses valeurs manquantes dans la plupart des datasets.

Dans ce dataset, beaucoup de colonnes prennent la valeur `NA` comme valeur attendue. Nous allons tout de même considérer ces valeurs comme de vraies valeurs manquantes, car comme nous allons le voir il y a beaucoup de colonnes pour lesquelles nous pouvons faire mieux que de laisser la catégorie `NA` en place.

Il est souvent intéressant de quantifier ce manque et de le pallier si cela est judicieux. Il est par exemple intéressant de noter que pour utiliser `sklearn`, il faut qu'aucune valeur ne manque pour que la quasi totalité des modèles fonctionne.

- *Avec la fonction [`pandas.DataFrame.isnull`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isnull.html), trouvez le nombre d'éléments manquants dans le training set.*
- *Adaptez ce procédé pour calculer le pourcentage d'éléments manquants par colonne.*
- *Triez ce résultat pour afficher en premier les colonnes auxquelles il manque le plus d'éléments. Vous pouvez utiliser [`DataFrame.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html?highlight=sort_values#pandas.DataFrame.sort_values)*
- *Effectuez aussi ce procédé pour le test set (d'autres nouvelles colonnes ont des valeurs manquantes).*
- *Affichez toutes les colonnes contenant au moins une valeur manquante (train set et test set confondus).*

In [ ]:
# Nombre d'éléments manquants total dans train_X
# Votre code ici

In [ ]:
# Pourcentage d'éléments manquant par colonne dans train_X
# Votre code ici

# Si ce n'est pas déjà fait, triez le résultat en décroissant
# Votre code ici

In [ ]:
# Nombre d'éléments manquants total dans test_X
# Votre code ici

In [ ]:
# Pourcentage d'éléments manquant par colonne dans test_X
# Votre code ici

# Si ce n'est pas déjà fait, triez le résultat en décroissant
# Votre code ici

In [ ]:
# Toutes les colonnes ayant des valeurs manquantes (train et test)
# Votre code ici

#### Solution

In [ ]:
print(
  "Nombre de valeurs manquantes dans le training set :", train_X.isnull().sum().sum()
)

In [ ]:
type(train_X.isnull().sum())

In [ ]:
def columns_with_missing_values(df: pandas.DataFrame) -> pandas.Series:
  df_na: pandas.Dataframe = (df.isnull().sum() / df.shape[0]) * 100
  df_na = df_na[df_na > 0]
  return df_na.sort_values(ascending=False)


def display_percentages_series(s: pandas.Series) -> None:
  print("\n".join([f"{name:>20.20} {value:5.2f}%" for name, value in s.items()]))


print("Pourcentage de valeur manquante par feature dans le train set :")
missing_train = columns_with_missing_values(train_X)
display_percentages_series(missing_train)

print("Pourcentage de valeur manquante par feature dans le test set :")
missing_test = columns_with_missing_values(test_X)
display_percentages_series(missing_test)

In [ ]:
[f"{name:20.20} {value:5.2f}%" for name, value in missing_train.items()]

In [ ]:
print("Nombre de valeurs manquantes dans le test set :", test_X.isnull().sum().sum())

In [ ]:
print("Colonnes comprenant des valeurs manquantes dans le training set ou le test set:")
print(", ".join(set(missing_train.index).union(missing_test.index)))
print()

print("Colonnes comprenant des valeurs manquantes seulement dans le training set:")
print(", ".join(set(missing_train.index).difference(missing_test.index)))
print()

print("Colonnes comprenant des valeurs manquantes seulement dans le test set:")
print(", ".join(set(missing_test.index).difference(missing_train.index)))

### Gestion des données manquantes

Il n'y a pas d'approche universelle du remplissage de valeur manquante. Chaque variable doit être traitée au cas par cas.

*En fonction de la documentation présente dans le fichier `dataset-ames/data_description.txt`, utilisez des groupes `cols_1` à `cols_4` pour appliquer 4 types de gestion des valeurs manquantes différents. Toutes ces gestions impliquent d'utiliser la fonction [`DataFrame.fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) qui permet de remplir les valeurs manquantes avec les arguments qu'on lui passe.*

*Chacun des groupes de colonnes sera associé à une méthode de remplacement des données manquantes*

*On pourra, par exemple, utiliser :*

- *la moyenne, la mediane ou 0 pour une variable réelle*
- *le mode pour une variable categorielle (valeur la plus présente)*
- *une classe "NA", surtout si elle est déjà existante*
- *…*

In [ ]:
# Votre code ici
cols_1 = ["Alley", "featureToto", "..."]
cols_2 = ["SaleType", "featureTiti", "..."]
# etc…

# all_X[cols_1] = all_X[cols_1].fillna("valeurParDefaut")

#### Solution

In [ ]:
# Remplissage avec la moyenne
cols_1 = ["LotFrontage", "GarageYrBlt"]
all_X.fillna(train_X[cols_1].mean(), inplace=True)

# Remplissages avec le mode
cols_2 = [
  "MSZoning",
  "Electrical",
  "KitchenQual",
  "Exterior1st",
  "Exterior2nd",
  "SaleType",
  "Utilities",
]
all_X.fillna(train_X[cols_2].mode().iloc[0, :], inplace=True)

# Remplissages avec 0
cols_4 = [
  "GarageArea",
  "GarageCars",
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtFullBath",
  "BsmtHalfBath",
  "BsmtUnfSF",
  "MasVnrArea",
  "TotalBsmtSF",
]
all_X[cols_4] = all_X[cols_4].fillna(0)

# Un remplissage spécifique
cols_5 = ["Functional"]
all_X[cols_5] = all_X[cols_5].fillna("Typ")

# On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
all_X = all_X.fillna("NA")

# On vérifie que l'on n'a rien oublié
print(all_X.isnull().sum().sum())

### Variables catégorielles, mais codées numériquement

Certaines catégorie telles que `MSSubClass` sont codées avec des codes numériques exprimant des catégories (cf. `data_description.txt`). Si on ne fait rien, cette variable va être traitée comme numérique alors que les catégories décrites ne sont pas même ordinales.


In [ ]:
# On transforme le codage numérique en string afin que ce soit traité comme
# une variable catégorielle
cols_numerical2label = ["MSSubClass"]
all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

### Variables ordinales

Pour ne pas perdre l'information d'ordre des variables ordinales, il est important d'utiliser le label encoding vu en cours.

- *Label encodez les variables `BsmtCond` et `FireplaceQu`. Il faudra utiliser la fonction [`DataFrame.replace`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html).*

In [ ]:
# #Création du dictionnaire de dictionnaire utilisé pour le remplacement des variables nominales en entiers
# #Key   : "Nom de la variable"
# #Value :  Dictionnaire de remplacement des modalités de la variable
# replace_mapping = dict(???)

# all_X = all_X.replace(???)

#### Solution

In [ ]:
# Création du dictionnaire de dictionnaire utilisé pour le remplacement des variables nominales en entiers
# Key   : "Nom de la variable"
# Value :  Dictionnaire de remplacement des modalités de la variable
replace_mapping = dict(
  Alley=dict(NA=0, Grvl=1, Pave=2),
  BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
  BsmtFinType1=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  BsmtFinType2=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
  LandSlope=dict(Sev=1, Mod=2, Gtl=3),
  LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
  PavedDrive=dict(NA=0, N=1, P=2, Y=3),
  Street=dict(Grvl=1, Pave=2),
  Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  BsmtCond=dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5),
)
# Toutes les variables suivantes peuvent utiliser le même dictionnaire de remplacement
quality_columns = [
  "BsmtCond",
  "BsmtQual",
  "ExterCond",
  "ExterQual",
  "FireplaceQu",
  "GarageCond",
  "GarageQual",
  "HeatingQC",
  "KitchenQual",
  "PoolQC",
]
quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
for quality_column in quality_columns:
  replace_mapping[quality_column] = quality_mapping

all_X.replace(replace_mapping, inplace=True)

# Cast explicite de ces variables en type entier
ordinal_columns = list(replace_mapping.keys())
all_X[ordinal_columns] = all_X[ordinal_columns].astype(int)

### One hot encoding des variables catégorielles

Maintenant que les variables ordinales sont traitées, il faut encore traiter les variables catégorielles.

- *Appliquez la transformation de one-hot encoding avec la fonction [`pandas.get_dummies`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html)*
- *Comment a évolué la forme de `all_X` ?*

In [ ]:
# all_X = ???

#### Solution

In [ ]:
print(all_X.shape)
all_X = pandas.get_dummies(all_X)
print(all_X.shape)

In [ ]:
print("Colonnes :", ", ".join(all_X.columns))

Chaque catégorie de chaque variable catégorielle a été transformée en feature. La matrice d'entrée est passée de sa taille originale de $\mathbb{R}^{2919 \times 79}$ à $\mathbb{R}^{2919 \times 244}$.

### Réassemblage

Maintenant que nous avons traité nos données, nous pouvons séparer `all_X` en `train_X` et `test_X` de nouveau.

*Utilisez la forme de `train_X` pour savoir à quel endroit séparer `all_X`.*

In [ ]:
# split_index = ???
# train_X = ???
# test_X = ???

#### Solution

In [ ]:
split_index = train_X.shape[0]
train_X = all_X.iloc[:split_index, :]
test_X = all_X.iloc[split_index:, :]

In [ ]:
train_X.shape

## Exploration des données

### Analyse de la variable de sortie

Les principales étapes de nettoyage des données étant finies, on peut commencer à analyser le dataset. Cette [page du manuel de seaborn](https://seaborn.pydata.org/tutorial/distributions.html#plotting-univariate-distributions) explique les principales techniques nécessaires.

*Utilisez `seaborn` pour visualiser la variable de sortie.*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
seaborn.displot(train_y, kde=True)
plt.show()

### Comparer la variable de sortie à une distribution normale

Un prérequis de beaucoup d'approches est la normalité de la variable de sortie.

*Testez cette hypothèse.*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
seaborn.distplot(train_y, fit=scipy.stats.norm)
plt.show()

fig = plt.figure()
scipy.stats.probplot(train_y, dist="norm", plot=plt)
plt.show()

### Analyse des corrélations à la variable de sortie

Maintenant que nous avons une bonne idée de la forme de la variable de sortie, analysons les variables qui lui sont le plus corrélées.

*Utilisez [`DataFrame.corr`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) et [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) pour afficher une matrice de corrélation. Optionnellement, triez la en fonction des corrélations avec la variable de sortie*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
target_corrs = (
  train_X
  # Calcul des corrélations avec la variable de sortie
  .corrwith(train_y)
  # Suppression des NaN créés : ils correspondent aux colonnes
  # constantes
  .dropna()
  # Tri par valeur de corrélation : les colonnes les plus corrélées
  # seront les premières, les plus anti-corrélées les dernières
  .sort_values(ascending=False)
)

# Récupération des dix premières et dix dernières valeurs
top_target_corrs = pandas.concat([target_corrs[:10], target_corrs[-10:]])
top_target_corrs.rename("SalePrice", inplace=True)

# Calcul des corrélations des variables les plus corrélées (et anti-corrélées) entre elles
feature_corrs = train_X.loc[:, top_target_corrs.index].corr()

# Regroupement des corrélations avec la cible et des corrélations entre
# variables
corrs = pandas.concat([top_target_corrs, feature_corrs], axis=1)

# Affichage de la matrice de corrélation à l'aide d'une heatmap
_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=-1, vmax=1, cmap=seaborn.diverging_palette(220, 20, n=100))
ax.set_title("Plus grandes corrélations et anti-corrélations")
plt.show()

In [ ]:
corrs

### Exploration des variables les plus corrélées

Les variables les plus corrélées sont maintenant connues. Il reste à nous renseigner sur ces dernières.

*Graphez la distribution jointe de `GrLivArea` et `SalePrice` ainsi que `OverallQual` et `SalePrice`.*

In [ ]:
# Votre code ici


#### Solution

In [ ]:
seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)
plt.show()
seaborn.violinplot(x=train_X["OverallQual"], y=train_y)
plt.show()

### Analyse des variables les plus corrélées

*Que remarquez-vous sur ces deux plots ?*

#### Réponse

Il semble qu'il y ait deux valeurs clairement outlier : les valeurs aux plus grandes `GrLivArea`.

Cependant, une analyse plus poussée est nécessaire pour savoir si elles le sont vraiment (erreurs dans le jeu de données) ou si ce sont juste des maisons aux caractéristiques inhabituelles (terres agricoles, maisons non terminées, etc).